<a href="https://colab.research.google.com/github/zaid9281/NLP-Anoja_samatrix/blob/main/assignment1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Name:Mohammad Zaid

Class: B.Tech CSE AI-ML

Sec-A

Roll no.: 2401730232

Anoja Samatrix

Section 1: Setup and Import

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass
from typing import List, Tuple

In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score


In [ ]:

# Download WordNet resource quietly for lemmatization
nltk.download('wordnet', quiet=True)

# Set global seed for exact reproducibility across runs
SEED = 101
random.seed(SEED)
np.random.seed(SEED)

Section 2: Structured Data set construction and storage

In [ ]:
@dataclass
class ProductDatasetBuilder:
    """Handles formatted dataset creation, shuffling, and local CSV persistence."""
    output_filename: Path = Path("product_sentiment_data.csv")

    def _get_corpus_samples(self) -> Tuple[List[str], List[str]]:
        """Returns 60 positive and 60 negative e-commerce product reviews."""
        positive_reviews = [
            "Outstanding quality! The build is durable and works flawlessly right out of the box.",
            "Fast shipping, excellent packaging, and the product exceeded my expectations.",
            "Great value for money. Highly reliable and easy to set up.",
            "The battery life is exceptional. Easily lasts two full days of heavy use.",
            "Fantastic customer support resolved my inquiry in minutes. Very satisfied!",
            "Crisp audio, clear display, and smooth performance overall. Top tier item.",
            "Sleek design and feels premium in hand. Would definitely buy again.",
            "Works as advertised without any issues. Five stars!",
            "Prompt delivery and impressive craftmanship. Worth every penny.",
            "Surprisingly lightweight yet sturdy. Highly recommended for daily use.",
            "Superb performance. Handled everything I threw at it seamlessly.",
            "Very intuitive controls and clear instruction manual provided.",
            "Affordable pricing for such high quality. Exceeded initial expectations.",
            "The screen brightness is superb even under direct sunlight.",
            "Flawless integration with my existing devices. Couldn't be happier.",
            "Beautiful aesthetic, ergonomic grip, and functions silently.",
            "Order arrived two days ahead of schedule in pristine condition.",
            "Exceptional software optimization and quick response times.",
            "Solid construction with premium metallic finish. Highly durable.",
            "The noise cancellation works like magic. Total game changer.",
            "Quick setup process took less than five minutes. Very user friendly.",
            "Robust materials used throughout. Feels like it will last years.",
            "Great attention to detail in design and functionality.",
            "Completely satisfied with this purchase. Outstanding service overall.",
            "Compact size yet powerful output. Exactly what I needed.",
            "The camera quality is crisp and vivid. Captures great detail.",
            "Smooth customer experience from checkout to delivery.",
            "Solid reliability after 3 months of continuous usage.",
            "Delivered with zero scratches or defects. Top quality assurance.",
            "Extremely comfortable to use over extended periods.",
            "Generous warranty terms and very helpful support team.",
            "The speed improvement compared to older models is massive.",
            "Elegant packaging makes it a perfect gift choice.",
            "Zero lag, pristine finish, and quick charging capabilities.",
            "Impressive feature set for this price category.",
            "High definition resolution with accurate color reproduction.",
            "Strong connectivity with no drops or connection issues.",
            "Well engineered, silent operation, and high efficiency.",
            "Delivered quickly with all original accessories included.",
            "The software updates keep adding valuable new functions.",
            "Top notch build quality and very responsive touch interface.",
            "Lightweight, durable, and highly portable. Perfect companion.",
            "The sound clarity is pristine across all volume levels.",
            "Fast charging support is a huge time saver.",
            "Effortless maintenance and easy to clean surface.",
            "Exceptional tactile response on all physical buttons.",
            "The companion app is clean, well designed, and responsive.",
            "Bright LED indicators and easy to read displays.",
            "Superior heat dissipation, stays cool during heavy use.",
            "Includes all necessary cables and mount accessories.",
            "Unbeatable performance per dollar spent. Very happy.",
            "Vibrant colors and high contrast ratio on the display panel.",
            "Comfortable fit and doesn't cause fatigue after long hours.",
            "The automatic synchronization features work like a charm.",
            "Sturdy hinges and smooth mechanism overall.",
            "Quick return policy reassurance, though I won't need it!",
            "Impeccable finish without any rough edges.",
            "High efficiency energy consumption ratings verified.",
            "Super quiet fan and excellent thermal management.",
            "A complete package that delivers on every single promise."
        ]

        negative_reviews = [
            "Poor quality materials. Broke within two weeks of light usage.",
            "Terrible customer service. Nobody responded to my refund requests.",
            "Overpriced junk. The item arrived damaged and non-functional.",
            "Battery drains within two hours. Completely useless for travel.",
            "Extremely slow shipping. Took over three weeks to arrive.",
            "The buttons are sticky and non-responsive. Poor quality control.",
            "Flimsy plastic construction that feels cheap and delicate.",
            "Constantly disconnects from Bluetooth. Unstable performance.",
            "Misleading product description. Features advertised are missing.",
            "The display is dim and hard to read under standard lighting.",
            "Overheats dangerously after just fifteen minutes of operation.",
            "Defective unit received out of the box. Extremely frustrating.",
            "Missing critical setup cables from the sealed package.",
            "Glitchy software crashes continuously. Unusable state.",
            "Very uncomfortable design causes strain after short usage.",
            "The audio output is flat, quiet, and filled with static noise.",
            "Waste of money. Stopped working entirely after a firmware update.",
            "Customer support refused to honor the warranty agreement.",
            "Scratched and smudged unit sent as new. Terrible experience.",
            "Extremely complicated setup instructions with missing steps.",
            "Loud buzzing noise emitted constantly during operation.",
            "Inferior durability compared to older versions of this item.",
            "The charger gets alarmingly hot during standard charging cycles.",
            "False claims regarding waterproof ratings. Water damaged immediately.",
            "Unstable connection drops repeatedly during active use.",
            "Poorly packaged, carton was crushed upon arrival.",
            "Screen suffers from noticeable back-light bleeding issues.",
            "The mobile app requires invasive permissions and crashes often.",
            "Weak construction leads to creaking sounds under pressure.",
            "Too bulky and heavy to carry around conveniently.",
            "Zero bass response and distorted high frequencies.",
            "Replacement parts are non-existent or impossible to purchase.",
            "Frequent latency issues make it useless for real-time tasks.",
            "The color does not match the online product images at all.",
            "Defective power button requires heavy pressure to register.",
            "Poor ergonomics lead to wrist strain during extended use.",
            "The device randomly reboots itself multiple times daily.",
            "Excessive plastic waste used in inner packaging.",
            "Non-standard charging port limits cable compatibility options.",
            "The touch panel is overly sensitive and registers phantom touches.",
            "Refused return despite sending back within the return window.",
            "Feels like a cheap knock-off product. Extremely dissatisfied.",
            "Software lacks basic features present in competing models.",
            "Slow file transfer speeds well below advertised rates.",
            "Rubber coating degrades and becomes sticky over short time.",
            "The power cord is far too short to reach standard outlets.",
            "Unreliable sensors produce inaccurate reading data.",
            "Excessive blooming and washed out colors on the panel.",
            "Zero response from technical support regarding bug fixes.",
            "Unit arrived dirty with visible fingerprints and dust.",
            "The latch mechanism is weak and pops open unexpectedly.",
            "Extremely noisy fan sounds like a vacuum cleaner.",
            "The stand is flimsy and causes the unit to wobble easily.",
            "Short circuit triggered on first plug-in. Safety hazard!",
            "Unclear diagnostic light indicators make troubleshooting impossible.",
            "Disappointing overall value given the premium price point.",
            "Microphone pickup is quiet and picks up excessive ambient noise.",
            "Firmware updates keep failing midway and bricking settings.",
            "Terrible build consistency between two ordered units.",
            "Not worth buying. Highly regret this purchase decision."
        ]
        return positive_reviews, negative_reviews

    def generate_and_save(self) -> pd.DataFrame:
        pos, neg = self._get_corpus_samples()
        records = []

        for i, (p_text, n_text) in enumerate(zip(pos, neg), start=1001):
            records.append({"item_id": f"ITEM-{i}", "review_body": p_text, "sentiment": "positive"})
            records.append({"item_id": f"ITEM-{i+500}", "review_body": n_text, "sentiment": "negative"})

        df = pd.DataFrame(records)
        # Shuffle dataset rows randomly
        df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        # Save as standard formatted text file (CSV)
        df.to_csv(self.output_filename, index=False)
        return df

Section 3: Dataset integrity and verfication

In [ ]:
def verify_dataset_structure(file_path: Path):
    """Loads CSV and outputs formatted structural diagnostics."""
    print("==================================================")
    print(" SECTION 3: DATASET STRUCTURE & INSPECTION")
    print("==================================================")

    # Read raw text lines directly from disk
    raw_lines = file_path.read_text(encoding="utf-8").splitlines()
    print(">>> Raw CSV Header + First 5 Records:")
    for line in raw_lines[:6]:
        print("   ", line)
    print(f"    ... Total lines saved in file: {len(raw_lines)}")

    # Load back using Pandas
    df = pd.read_csv(file_path)
    print("\n>>> DataFrame Diagnostics:")
    print(f"    Dimensions (Rows, Columns) : {df.shape}")
    print(f"    Missing Values             : {df.isnull().sum().sum()}")
    print(f"    Target Balance:\n{df['sentiment'].value_counts().to_string()}\n")

Section 4: Text processing and normalisation

In [ ]:
class TextNormalizer:
    """Standardizes text via lowercasing, contraction expansion, stopword filtering, and lemmatization."""
    def __init__(self):
        self.lemmatizer = WordNetLemmatizer()
        # Retain negation keywords to avoid losing critical polarity context
        self.negation_words = {"not", "no", "never", "nor", "neither", "cannot", "n't"}
        self.stop_words = set(ENGLISH_STOP_WORDS) - self.negation_words

    def transform(self, text: str) -> str:
        # Lowercase text
        text = str(text).lower()

        # Expand common English contractions
        text = re.sub(r"won\'t", "will not", text)
        text = re.sub(r"can\'t", "cannot", text)
        text = re.sub(r"n\'t", " not", text)
        text = re.sub(r"\'re|\'ve|\'ll|\'d|\'m|\'s", "", text)

        # Remove non-alphabetical punctuation and numbers
        text = re.sub(r"[^a-z\s]", " ", text)

        # Tokenize and remove stopwords
        tokens = [t for t in text.split() if t not in self.stop_words and len(t) > 1]

        # Lemmatization
        lemmas = [self.lemmatizer.lemmatize(token) for token in tokens]
        return " ".join(lemmas)

Section 5: Feature Engineering

In [ ]:
def perform_feature_engineering(df: pd.DataFrame):
    """Demonstrates Bag-of-Words (Count) and TF-IDF matrix generation."""
    print("==================================================")
    print(" SECTION 5: FEATURE EXTRACTION & ENGINEERING")
    print("==================================================")

    # 1. Bag of Words Vectorization
    bow_vec = CountVectorizer()
    bow_matrix = bow_vec.fit_transform(df["processed_text"])
    print(">>> Bag-of-Words (CountVectorizer):")
    print(f"    Feature Matrix Shape : {bow_matrix.shape}")
    print(f"    Vocabulary Sample    : {list(bow_vec.get_feature_names_out()[:8])}")

    # 2. TF-IDF Vectorization with Unigrams + Bigrams
    tfidf_vec = TfidfVectorizer(ngram_range=(1, 2))
    tfidf_matrix = tfidf_vec.fit_transform(df["processed_text"])
    print("\n>>> TF-IDF (Unigrams + Bigrams):")
    print(f"    Feature Matrix Shape : {tfidf_matrix.shape}")
    print(f"    Sample N-grams       : {list(tfidf_vec.get_feature_names_out()[20:26])}\n")

# Section 6: Model training

In [ ]:
class SklearnTextCleaner(BaseEstimator, TransformerMixin):
    """Scikit-Learn wrapper around TextNormalizer for pipeline inclusion."""
    def __init__(self):
        self.normalizer = TextNormalizer()

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return [self.normalizer.transform(text) for text in X]


def run_pipeline(df: pd.DataFrame):
    """Assembles and evaluates the end-to-end NLP classification pipeline."""
    print("==================================================")
    print(" SECTION 6: PIPELINE ASSEMBLY & EVALUATION")
    print("==================================================")

    # Train-Test Split (80% Train, 20% Test)
    X_train, X_test, y_train, y_test = train_test_split(
        df["review_body"],
        df["sentiment"],
        test_size=0.20,
        random_state=SEED,
        stratify=df["sentiment"]
    )

    # Scikit-Learn Pipeline: Preprocessing -> TF-IDF -> Classifier
    full_pipeline = Pipeline([
        ('cleaner', SklearnTextCleaner()),
        ('vectorizer', TfidfVectorizer(ngram_range=(1, 2))),
        ('classifier', LogisticRegression(C=1.0, random_state=SEED))
    ])

    # Fit pipeline
    full_pipeline.fit(X_train, y_train)

    # Test evaluation
    y_pred = full_pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    print(f">>> Split: {len(X_train)} Train records | {len(X_test)} Test records")
    print(f">>> Pipeline Model Accuracy: {accuracy * 100:.2f}%\n")
    print(">>> Confusion Matrix (Rows = Actual, Cols = Predicted):")
    print(confusion_matrix(y_test, y_pred, labels=["negative", "positive"]))
    print("\n>>> Classification Metric Report:")
    print(classification_report(y_test, y_pred))

    return full_pipeline

# Section 7: Inference and sentification predication

In [ ]:
def predict_new_feedback(pipeline: Pipeline):
    """Tests the fully trained pipeline on completely unseen inputs."""
    print("==================================================")
    print(" SECTION 7: INFERENCE ON UNSEEN SAMPLES")
    print("==================================================")

    unseen_texts = [
        "The camera produces brilliant high resolution shots, love it!",
        "Device arrived broken, battery life is weak and unusable.",
        "Not bad at all, works much better than my older device."
    ]

    predictions = pipeline.predict(unseen_texts)
    confidence_scores = pipeline.predict_proba(unseen_texts).max(axis=1)

    for text, label, score in zip(unseen_texts, predictions, confidence_scores):
        print(f"[{label.upper():8s} | Confidence: {score:.1%}]  \"{text}\"")
    print("==================================================")

Main Execution

In [ ]:
if __name__ == "__main__":
    # Question 1: Dataset Build
    builder = ProductDatasetBuilder()
    df_raw = builder.generate_and_save()
    verify_dataset_structure(builder.output_filename)

    # Preprocess text column for feature engineering demo
    normalizer = TextNormalizer()
    df_raw["processed_text"] = df_raw["review_body"].apply(normalizer.transform)

    # Question 2: Features & Pipeline
    perform_feature_engineering(df_raw)
    trained_model = run_pipeline(df_raw)

    # Inference Demonstration
    predict_new_feedback(trained_model)

 SECTION 3: DATASET STRUCTURE & INSPECTION
>>> Raw CSV Header + First 5 Records:
    item_id,review_body,sentiment
    ITEM-1507,Flimsy plastic construction that feels cheap and delicate.,negative
    ITEM-1035,Impressive feature set for this price category.,positive
    ITEM-1028,Solid reliability after 3 months of continuous usage.,positive
    ITEM-1511,Overheats dangerously after just fifteen minutes of operation.,negative
    ITEM-1534,The color does not match the online product images at all.,negative
    ... Total lines saved in file: 121

>>> DataFrame Diagnostics:
    Dimensions (Rows, Columns) : (120, 3)
    Missing Values             : 0
    Target Balance:
sentiment
negative    60
positive    60

 SECTION 5: FEATURE EXTRACTION & ENGINEERING
>>> Bag-of-Words (CountVectorizer):
    Feature Matrix Shape : (120, 476)
    Vocabulary Sample    : ['accessory', 'accurate', 'active', 'adding', 'advertised', 'aesthetic', 'affordable', 'agreement']

>>> TF-IDF (Unigrams + Bigrams):
  